#Transforms orders data

In [0]:
df = spark.table('gizmobox.bronze.v_orders')
display(df)

In [0]:
from pyspark.sql import functions as F
df1 = df.select(F.regexp_replace("value", '"order_date": (\\d{4}-\\d{2}-\\d{2})','"order_date": "$1"').alias('fixed_value'))
display(df1)

##Extracting json schema

In [0]:
df_temp = df1.select(F.schema_of_json(F.col("fixed_value")).alias("schema"))
display(df_temp.limit(1))

In [0]:
order_schema = '''STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>'''

##converting json string to json objects

In [0]:
df2 = df1.select(F.from_json("fixed_value", order_schema).alias("json_value"))
display(df2)

In [0]:
df2.writeTo('gizmobox.silver.py_order_json').createOrReplace()

##Explode Arrays

In [0]:
df_orders = spark.table('gizmobox.silver.py_order_json')
display(df_orders)

In [0]:
from pyspark.sql import functions as F
df_orders_norm = df_orders.select(
    "json_value.order_id",
    "json_value.order_status",
    "json_value.payment_method",
    "json_value.total_amount",
    "json_value.transaction_timestamp",
    "json_value.items",
    "json_value.customer_id"
) 
display(df_orders_norm)

##array duplicates

In [0]:
df_orders_norm = df_orders.select(
    "json_value.order_id",
    "json_value.order_status",
    "json_value.payment_method",
    "json_value.total_amount",
    "json_value.transaction_timestamp",
    "json_value.customer_id",
    F.array_distinct("json_value.items").alias("items")
) 
display(df_orders_norm)

##Explode arrays

In [0]:
df_orders_exploded = df_orders_norm.select (
    "order_id",
    "order_status",
    "payment_method",
    "total_amount",
    "transaction_timestamp",
    "customer_id",
    F.explode("items").alias("item")
)
display(df_orders_exploded)

In [0]:
df_orders_exploded_1 = df_orders_exploded.select (
    "order_id",
    "order_status",
    "payment_method",
    "total_amount",
    "transaction_timestamp",
    "customer_id",
    "item.item_id",
    "item.category",
    "item.name",
    "item.price",
    "item.quantity",
    "item.details.brand",
    "item.details.color"
)
display(df_orders_exploded_1)

In [0]:
df_orders_exploded_1.writeTo('gizmobox.silver.py_orders').createOrReplace()